# 01. CSE Banking Data Collection

**Tickers**: COMB.N0000.LK, HNB.N0000.LK, NDB.N0000.LK, BFLN.N0000.LK, DFCC.N0000.LK, SAMP.N0000.LK

**Sources**: CSE API → Twelve Data → Alpha Vantage → Sample Data (fallback)  
**Cloud**: AWS S3 upload

**Status**: 🔄 Implementing CSE API Integration

In [6]:
import sys
import os
sys.path.append('..')  # Add parent directory to path

from cse_api_client import CSEDataClient
import pandas as pd
from datetime import datetime
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Sri Lankan Banking Tickers
BANKING_TICKERS = [
    'COMB.N0000.LK',  # Commercial Bank of Ceylon
    'HNB.N0000.LK',   # Hatton National Bank
    'NDB.N0000.LK',   # National Development Bank
    'DFCC.N0000.LK',  # DFCC Bank
    'SAMP.N0000.LK',  # Sampath Bank
    'BFLN.N0000.LK'   # Bank of Ceylon
]

def main():
    """Main data collection function"""
    print("🚀 Starting CSE Banking Data Collection")
    print("=" * 50)

    # Initialize CSE API client
    client = CSEDataClient()

    # Date range: 2 years back for sufficient historical data
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')

    print(f"📅 Date Range: {start_date} to {end_date}")
    print(f"🏦 Tickers: {', '.join(BANKING_TICKERS)}")
    print()

    # Fetch data for all tickers
    print("📡 Fetching data from CSE API with fallbacks...")
    results = client.fetch_multiple_tickers(BANKING_TICKERS, start_date, end_date)

    if not results:
        print("❌ No data could be fetched from any source")
        return

    # Combine all ticker data into multi-index DataFrame (yfinance format)
    print("🔄 Processing and combining data...")
    combined_data = {}
    summary_stats = []

    for ticker, df in results.items():
        if df is not None and not df.empty:
            # Validate data quality
            if client.validate_data(df, ticker):
                combined_data[ticker] = df

                # Calculate summary statistics
                latest_price = df['Close'].iloc[-1] if not df.empty else 0
                avg_volume = df['Volume'].mean() if not df.empty else 0
                data_points = len(df)

                summary_stats.append({
                    'Ticker': ticker,
                    'Latest Price': f"${latest_price:.2f}",
                    'Data Points': data_points,
                    'Avg Volume': f"{avg_volume:,.0f}",
                    'Date Range': f"{df.index.min().date()} to {df.index.max().date()}"
                })

                print(f"✅ {ticker}: {data_points} days, latest: ${latest_price:.2f}")
            else:
                print(f"❌ {ticker}: Data validation failed")
        else:
            print(f"❌ {ticker}: No data available")

    if not combined_data:
        print("❌ No valid data to save")
        return

    # Create multi-index DataFrame (compatible with existing analysis)
    try:
        final_df = pd.concat(combined_data, axis=1, keys=combined_data.keys())
        print(f"\n📊 Combined data shape: {final_df.shape}")

        # Save to CSV
        output_path = '../data/banking_ohlcv_raw.csv'
        final_df.to_csv(output_path)
        print(f"💾 Saved to {output_path}")

        # Display summary table
        print("\n📈 DATA COLLECTION SUMMARY")
        print("=" * 60)
        summary_df = pd.DataFrame(summary_stats)
        print(summary_df.to_string(index=False))

        print(f"\n✅ Successfully collected data for {len(combined_data)}/{len(BANKING_TICKERS)} tickers")
        print("🎯 Ready for preprocessing and analysis!")
        return final_df




    except Exception as e:
        print(f"❌ Error processing final data: {e}")
        # Save individual ticker files as fallback
        for ticker, df in combined_data.items():
            try:
                df.to_csv(f'../data/{ticker.replace(".N0000.LK", "")}_data.csv')
                print(f"💾 Saved individual file: {ticker}")
            except Exception as e2:
                print(f"❌ Failed to save {ticker}: {e2}")

if __name__ == "__main__":
    df = main()

INFO:cse_api_client:CSE Data Client initialized
INFO:cse_api_client:Fetching data for COMB.N0000.LK from 2024-03-21 to 2026-03-21
INFO:cse_api_client:Attempting to fetch COMB.N0000.LK from CSE direct API
INFO:cse_api_client:Fetching COMB.N0000.LK from Twelve Data


🚀 Starting CSE Banking Data Collection
📅 Date Range: 2024-03-21 to 2026-03-21
🏦 Tickers: COMB.N0000.LK, HNB.N0000.LK, NDB.N0000.LK, DFCC.N0000.LK, SAMP.N0000.LK, BFLN.N0000.LK

📡 Fetching data from CSE API with fallbacks...


INFO:cse_api_client:Fetching COMB.N0000.LK from Alpha Vantage
INFO:cse_api_client:Generating sample data for COMB.N0000.LK
INFO:cse_api_client:Generated 731 days of sample data for COMB.N0000.LK
INFO:cse_api_client:Successfully fetched data for COMB.N0000.LK
INFO:cse_api_client:Fetching data for HNB.N0000.LK from 2024-03-21 to 2026-03-21
INFO:cse_api_client:Attempting to fetch HNB.N0000.LK from CSE direct API
INFO:cse_api_client:Fetching HNB.N0000.LK from Twelve Data
INFO:cse_api_client:Fetching HNB.N0000.LK from Alpha Vantage
INFO:cse_api_client:Generating sample data for HNB.N0000.LK
INFO:cse_api_client:Generated 731 days of sample data for HNB.N0000.LK
INFO:cse_api_client:Successfully fetched data for HNB.N0000.LK
INFO:cse_api_client:Fetching data for NDB.N0000.LK from 2024-03-21 to 2026-03-21
INFO:cse_api_client:Attempting to fetch NDB.N0000.LK from CSE direct API
INFO:cse_api_client:Fetching NDB.N0000.LK from Twelve Data
INFO:cse_api_client:Fetching NDB.N0000.LK from Alpha Vantage

🔄 Processing and combining data...
✅ COMB.N0000.LK: 731 days, latest: $91.80
✅ HNB.N0000.LK: 731 days, latest: $48.51
✅ NDB.N0000.LK: 731 days, latest: $37.98
✅ DFCC.N0000.LK: 731 days, latest: $155.12
✅ SAMP.N0000.LK: 731 days, latest: $23.01
✅ BFLN.N0000.LK: 731 days, latest: $106.08

📊 Combined data shape: (731, 36)
💾 Saved to ../data/banking_ohlcv_raw.csv

📈 DATA COLLECTION SUMMARY
       Ticker Latest Price  Data Points Avg Volume               Date Range
COMB.N0000.LK       $91.80          731    105,375 2024-03-21 to 2026-03-21
 HNB.N0000.LK       $48.51          731    104,409 2024-03-21 to 2026-03-21
 NDB.N0000.LK       $37.98          731    105,403 2024-03-21 to 2026-03-21
DFCC.N0000.LK      $155.12          731    107,933 2024-03-21 to 2026-03-21
SAMP.N0000.LK       $23.01          731    105,441 2024-03-21 to 2026-03-21
BFLN.N0000.LK      $106.08          731    106,205 2024-03-21 to 2026-03-21

✅ Successfully collected data for 6/6 tickers
🎯 Ready for preprocessing and an

In [ ]:
import os
import boto3
from io import StringIO
from dotenv import load_dotenv

# Load credentials from .env file
load_dotenv()

# Initialize the S3 client
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION', 'ap-south-1') 
)

# newly created bucket!
bucket_name = 'srilanka-banking-trading-raw-data-dj2901' 

try:
    # Convert the pandas dataframe to a CSV string buffer
    csv_buffer = StringIO()
    df.to_csv(csv_buffer)

    # Upload the buffer to S3
    s3_client.put_object(
        Bucket=bucket_name, 
        Key='raw_cse_banking_data.csv', 
        Body=csv_buffer.getvalue()
    )
    print(f"✅ Successfully uploaded raw data to S3 bucket: {bucket_name}")
    
except Exception as e:
    print(f"❌ Failed to upload to S3. Error: {e}")

✅ Successfully uploaded raw data to S3 bucket: srilanka-banking-trading-raw-data-dj2901
